# Introduction

We are now moving to the final part of the workshop, which involves formulating business recommendations. Our tasks are:
- Determining a global betting odds,
- Dividing the dataset into categories: A, B, C, D, where A is the best group and D is the weakest group,
- Determining the risk of odds based on accepted parameters for each category.

As the last task, in a discussion format, we must consider the fact that we are a new betting company. When formulating our recommendations, we need to identify the risks that may affect our operations. We will perform this task together in a brainstorming session.

# Notebook Configuration

## Import necessary libraries

In [1]:
import pandas as pd

## Loading data into the workspace

> Remember to correctly specify the column separator

In [3]:
df = pd.read_csv(r'C:\Users\stefa\Desktop\CodersLab - Pyton - DA\Workshop - Aleksandra\Data\processed/hockey_teams.csv', sep=';', decimal='.')

### Checking data loading accuracy

In [4]:
df.head()

,team,season,victories,defeats,overtime_defeats,victory_percentage,scored_goals,received_goals,goal_difference,goals_ratio
0,Boston Bruins,1990,44,24,0,0.550,299,264,35,1.132576
1,Buffalo Sabres,1990,31,30,0,0.388,292,278,14,1.050360
2,Calgary Flames,1990,46,26,0,0.575,344,263,81,1.307985
3,Chicago Blackhawks,1990,49,23,0,0.613,284,211,73,1.345972
4,Detroit Red Wings,1990,34,38,0,0.425,273,298,-25,0.916107


# Determining Betting Odds

Let's review the content of the page: [click](https://trustbet.pl/kursy-bukmacherskie/), where information about methods for determining betting odds can be found. First, we will determine a global odd, which will be the starting point for our analysis (the so-called _baseline scenario_). At this point, we ignore the margin and assume that we are calculating the decimal odd.

Here is the list of steps to be performed to obtain the desired value:
- we will complete the definition of the `get_betting_odds` function, which will take `probability` of a given event as a parameter. We will use it multiple times, so it is worth preparing its implementation now
- then we need to appropriately aggregate the set and determine the **global** probability of the team's victory.

## Implementations of the `get_betting_odds` function

In [6]:
def get_betting_odds(probability):
    bet = 1/probability
    return bet
get_betting_odds(2)

0.5

### Some tests to check the correctness of the implementation

In [9]:
def test_get_betting_odds():
    assert get_betting_odds(1) == 1, "Expected 1"
    assert get_betting_odds(0.5) == 2, "Expected 2"
    assert get_betting_odds(0.25) == 4, "Expected 4"
    assert get_betting_odds(0.1) == 10, "Expected 10"
    try:
        get_betting_odds(0)
    except ZeroDivisionError:
        pass
    else:
        assert False, "Expected ZeroDivisionError"

    print("All tests passed!")

test_get_betting_odds()

All tests passed!


### Determining the global odds

Here, determine the probability of any team winning

In [18]:
gen_prob = float(round(df['victory_percentage'].mean(),3))
gen_prob

0.459

Set the global rate here using the `get_betting_odds` function. Round the result to two decimal places.

In [21]:
round(get_betting_odds(gen_prob),2)

2.18

# Team Categorization

Let's discuss how we can classify teams into _leagues_. We want to establish 4 leagues:
- A - league consisting of the best teams,
- B - league consisting of good teams,
- C - league consisting of average teams,
- D - league consisting of the weakest teams.

The above terms are quite subjective, so for the purpose of this exercise, we will adopt the following assumptions:
- A - the top 5% of teams,
- B - teams performing better than 70% of the group but worse than league A,
- C - teams performing better than 20% of the group but worse than league B,
- D - the remaining teams.

To accomplish this task, we will additionally implement the function `assign_team_to_league`.

> Note: This task looks unassuming, but it is difficult. Remember that during the class, you have access to the instructor, and later to a mentor.

## Determination of cutoff points for individual leagues

In [22]:
q95 = df['victory_percentage'].quantile(0.95)
q70 = df['victory_percentage'].quantile(0.70)
q20 = df['victory_percentage'].quantile(0.20)
def assign_team_to_league(x):
    if x > q95:
        return 'A'
    elif x > q70:
        return 'B'
    elif x > q20:
        return 'C'
    else:
        return 'D'

In [30]:
print("Test for q95:", assign_team_to_league(0.9))
print("Test for q70:", assign_team_to_league(0.6))
print("Test for q20:", assign_team_to_league(0.45))
print("Test for beton liga: :", assign_team_to_league(0.15))

Test for q95: A
Test for q70: B
Test for q20: C
Test for beton liga: : D


## Determination of odds per league

Here we set the betting odds for each league, which will allow us to draw final conclusions and establish the basic odds for individual teams.

> Remember: After generating the results, it is worth checking if they are reasonable.

In [31]:
df['league'] = df['victory_percentage'].apply(assign_team_to_league)
df.head()

,team,season,victories,defeats,overtime_defeats,victory_percentage,scored_goals,received_goals,goal_difference,goals_ratio,league
0,Boston Bruins,1990,44,24,0,0.550,299,264,35,1.132576,B
1,Buffalo Sabres,1990,31,30,0,0.388,292,278,14,1.050360,C
2,Calgary Flames,1990,46,26,0,0.575,344,263,81,1.307985,B
3,Chicago Blackhawks,1990,49,23,0,0.613,284,211,73,1.345972,B
4,Detroit Red Wings,1990,34,38,0,0.425,273,298,-25,0.916107,C


In [34]:
df.groupby('league')['victory_percentage'].mean()
#verovatnoca za pobedu po ligama

league
A    0.642500
B    0.559042
C    0.450479
D    0.308831
Name: victory_percentage, dtype: float64

Zasto ne mozemo da damo savet:
Trendovi su pokazali da je manje-vise liga izbalansirana u smislu, losiji timovi postaju bolji, a vrh se stabilizuje - nema posebno drasticnih promena.
Takodje, fali nam mnogo informacija npr:povrede kljucnih igraca, forma golmana, raspored utakmica, medjusobni skor, koji igrac kada ulazi u igru i zasto i mnogo, mnogo drugih faktora drugih faktora...
Svakako House always wins.

# Discussion

We have obtained certain odds values for each league. But how does this translate into real business? The entire task was about determining certain values from which a bookmaker can begin operations. Correct determination of these values is critical to attract customers to place bets with us, and on the other hand, inappropriate determination may lead to financial losses in the first days of operation.

For this reason, before translating the results and recommendations into business objectives, the analysis is subjected to discussion. Therefore, we will now take on a review role and would like to verify the steps. To that end, we will collectively discuss and critique our work by answering the following questions together:
- What elements of the analysis were simplified? What was omitted in the analysis?
- Are there any inconsistencies in the estimated odds? What are they?
- How can we improve the odds estimates?
- How can we enrich our initial dataset to make the estimates more accurate and less risky?
- How can we simulate the outcomes of our analysis to verify that they do not lead to financial losses?

This is a discussion panel, and every idea is valuable here.
